Ensure the music folder is not empty and press Run ALL.

In [14]:
import os
import numpy as np
import librosa
import json
import pandas as pd
from tensorflow.keras.models import load_model

# PATHS
MODEL_PATH = os.path.join('..', 'model', 'emotify_crnn_pro_final_80.h5')
LABELS_PATH = os.path.join('..', 'model', 'labels_map.json')
INPUT_FOLDER = os.path.join('..', 'data', 'New_Audio/')
SEGMENTS_CSV = os.path.join('..', 'results','segments_analysis.csv')

# 1. Model Loading
model = load_model(MODEL_PATH)
with open(LABELS_PATH, 'r') as f:
    labels = json.load(f)

def analyze_segment(y, sr):
    n_mels, target_steps = 128, 1292
    S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=n_mels, fmax=8000)
    S_dB = librosa.power_to_db(S, ref=np.max)
    S_dB = librosa.util.fix_length(S_dB, size=target_steps, axis=1)
    d_min, d_max = S_dB.min(), S_dB.max()
    if d_max - d_min != 0:
        S_dB = (S_dB - d_min) / (d_max - d_min)
    X = S_dB.reshape(1, 128, target_steps, 1)
    return model.predict(X, verbose=0)[0]

# 2. ANALYSING AUDIO FILES
all_segments_data = []
mp3_files = [f for f in os.listdir(INPUT_FOLDER) if f.endswith('.mp3')]

print(f"Analysing {len(mp3_files)} songs...")

for filename in mp3_files:
    path = os.path.join(INPUT_FOLDER, filename)
    try:
        full_duration = librosa.get_duration(path=path)
        segment_len = 30 
        
        for start in range(0, int(full_duration), segment_len):
            if start + segment_len > full_duration: break 
            
            y, sr = librosa.load(path, sr=22050, offset=start, duration=segment_len)
            scores = analyze_segment(y, sr)
            
            # Save segment info and scores
            segment_info = {
                'filename': filename,
                'start_sec': start,
                'end_sec': start + segment_len
            }
            # Add scores for each label
            for i, label in enumerate(labels):
                segment_info[label] = float(scores[i])
            
            all_segments_data.append(segment_info)
            
        print(f"✅ Completed segments for: {filename}")
    except Exception as e:
        print(f"❌ Error at {filename}: {e}")

# Save to CSV
df_segments = pd.DataFrame(all_segments_data)
df_segments.to_csv(SEGMENTS_CSV, index=False)
print(f"\n💾 All segments saved to '{SEGMENTS_CSV}'")

Analysing 5 songs...
✅ Completed segments for: ABBA - Voulez-Vous.mp3
✅ Completed segments for: All The Things She Said.mp3
✅ Completed segments for: Anita Ward - Ring My Bell.mp3
✅ Completed segments for: Love In Portofino.mp3
✅ Completed segments for: Master of Puppets (Remastered).mp3

💾 All segments saved to '..\results\segments_analysis.csv'


In [15]:
import pandas as pd
import numpy as np

# Optional: Set a minimum confidence threshold for the second emotion
MIN_CONFIDENCE = 0.30 


df = pd.read_csv(SEGMENTS_CSV)
emotion_labels = ['energy_joy', 'calm_relax', 'sadness_nostalgia', 'tension_fear']

# 1. Grouping and Averaging
df_total = df.groupby('filename')[emotion_labels].mean().reset_index()

results_summary = []

print(f"🏆 FINAL RESULTS (Threshold: {MIN_CONFIDENCE*100}%)")
print("="*75)

for index, row in df_total.iterrows():
    filename = row['filename']
    scores = [(label, row[label]) for label in emotion_labels]
    sorted_scores = sorted(scores, key=lambda x: x[1], reverse=True)
    
    top1_label, top1_val = sorted_scores[0]
    top2_label, top2_val = sorted_scores[1]
    
    # Filter based on the minimum confidence threshold
    display_text = f"🥇 Top 1: {top1_label:18s} ({top1_val*100:5.1f}%)"
    
    # Check if the Top 2 passes the threshold
    if top2_val >= MIN_CONFIDENCE:
        display_text += f" | 🥈 Top 2: {top2_label:18s} ({top2_val*100:5.1f}%)"
        final_top2 = top2_label
        final_top2_score = round(top2_val * 100, 2)
    else:
        display_text += f" | 🥈 (No strong second emotion found)"
        final_top2 = "None"
        final_top2_score = 0.0

    print(f"🎵 {filename[:30]:<30} | {display_text}")
    
    results_summary.append({
        'filename': filename,
        'top_1': top1_label,
        'top_1_score': round(top1_val * 100, 2),
        'top_2': final_top2,
        'top_2_score': final_top2_score
    })

# Save results
df_final = pd.DataFrame(results_summary)
df_final.to_csv(os.path.join('..','final_filtered_results.csv'), index=False)

🏆 FINAL RESULTS (Threshold: 30.0%)
🎵 ABBA - Voulez-Vous.mp3         | 🥇 Top 1: energy_joy         ( 83.8%) | 🥈 (No strong second emotion found)
🎵 All The Things She Said.mp3    | 🥇 Top 1: sadness_nostalgia  ( 43.1%) | 🥈 (No strong second emotion found)
🎵 Anita Ward - Ring My Bell.mp3  | 🥇 Top 1: energy_joy         ( 68.5%) | 🥈 (No strong second emotion found)
🎵 Love In Portofino.mp3          | 🥇 Top 1: calm_relax         ( 84.5%) | 🥈 Top 2: sadness_nostalgia  ( 70.1%)
🎵 Master of Puppets (Remastered) | 🥇 Top 1: energy_joy         ( 82.1%) | 🥈 (No strong second emotion found)


In [16]:
import pandas as pd

# 1. Loading the final results
FINAL_CSV = os.path.join('..', 'results', 'final_filtered_results.csv')
df = pd.read_csv(FINAL_CSV)
total_songs = len(df)

# 2. Claculating Top-1 Counts and Dominant Mood
# Top-1 counts and percentages
top1_counts = df['top_1'].value_counts()
top1_perc = (top1_counts / total_songs) * 100

# Most frequent Top-1 mood
dominant_mood = top1_counts.idxmax()

# 3. Function to generate advice based on the dominant mood and specific combinations
def generate_playlist_advice(mood, data):
    # Check for specific combinations of moods that may indicate emotional states
    numbness_count = len(data[(data['top_1'] == 'sadness_nostalgia') & (data['top_2'] == 'calm_relax')])
    distress_count = len(data[(data['top_1'] == 'sadness_nostalgia') & (data['top_2'] == 'tension_fear')])
    
    if distress_count >= 2: # If there are 2 or more instances of distressing combinations
        return "🆘 Caution: The playlist shows signs of significant internal turmoil. It is recommended to listen to more rhythmic and cheerful tracks for relief."

    if numbness_count >= 2:
        return "⚠️ WARNING: Signs of emotional numbness detected.\n Could the music be pulling you down instead of lifting you up? Try something with more energy."

    # General advice based on the dominant mood
    advice_map = {
        'energy_joy': "🌟 Your mood seems fantastic!\n Your music reflects vitality and positivity.",
        'calm_relax': "🧘 You are in a phase of calm and inner balance.\n Keep it up!",
        'sadness_nostalgia': "☁️ The playlist is dominated by melancholy.\n Reflection is healthy, but be careful not to isolate yourself.",
        'tension_fear': "⚡ Significant tension was detected.\n Could your daily stress be affecting your choices? Try something more soothing."
    }
    return advice_map.get(mood, "✅ Your music shows a healthy emotional diversity.")

# 4. Printing the final report
print("🧠 AI MENTAL HEALTH ADVISOR - FINAL REPORT")
print("="*60)
print(f"Analysed : {total_songs} songs")
print("-" * 60)

print(f"🏆 MOST DOMINANT MOOD: {dominant_mood.upper()}")
print(f"📈 Percentage of appearance: {top1_perc[dominant_mood]:.1f}%")
print("-" * 60)

print(f"💬 ADVISE FOR THE USER:")
print(generate_playlist_advice(dominant_mood, df))
print("="*60)

# Display the percentages of all categories
print("\n📊 ANALYTICAL REPORT (Top-1):")
for mood, perc in top1_perc.items():
    print(f"- {mood:18s}: {perc:4.1f}%")

🧠 AI MENTAL HEALTH ADVISOR - FINAL REPORT
Analysed : 5 songs
------------------------------------------------------------
🏆 MOST DOMINANT MOOD: ENERGY_JOY
📈 Percentage of appearance: 60.0%
------------------------------------------------------------
💬 ADVISE FOR THE USER:
🌟 Your mood seems fantastic!
 Your music reflects vitality and positivity.

📊 ANALYTICAL REPORT (Top-1):
- energy_joy        : 60.0%
- sadness_nostalgia : 20.0%
- calm_relax        : 20.0%
